In [1]:
import pandas as pd
import numpy as np

# Evaluation Wording Only

In [2]:
import os
import pandas as pd

# dynamic path
notebook_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_dir, ".."))

In [3]:
from scipy.spatial.distance import jensenshannon

In [4]:
def calculate_jsd_row(row):
    # get llm distribuations
    if pd.isna(row['LLM_Binary_Distribution']):
        return np.nan
    p = np.array([float(x) for x in row['LLM_Binary_Distribution'].split(',')])
    
    # get allbus distribuation
    pi_a = row['pi_ALLBUS']
    q = np.array([1.0 - pi_a, pi_a])

    js_metric = jensenshannon(p, q, base=2)
    return js_metric ** 2

In [5]:
def key_results_wording(dateiname):
    csv_path = os.path.join(repo_root, "results", dateiname)

    df_results_wording = pd.read_csv(csv_path, index_col="allbus_variable_variation")

    bias_v1 = df_results_wording['Bias_Direct'].iloc[0::3]
    bias_v2 = df_results_wording['Bias_Direct'].iloc[1::3]
    bias_v3 = df_results_wording['Bias_Direct'].iloc[2::3]
    
    mae_v1 = bias_v1.abs().mean()
    mae_v2 = bias_v2.abs().mean()
    mae_v3 = bias_v3.abs().mean()
    
    
    rmse_v1 = np.sqrt((bias_v1 ** 2).mean())
    rmse_v2 = np.sqrt((bias_v2 ** 2).mean())
    rmse_v3 = np.sqrt((bias_v3 ** 2).mean())

    df_results_wording['JSD_to_Human'] = df_results_wording.apply(calculate_jsd_row, axis=1)

    jsd_v1 = df_results_wording['JSD_to_Human'].iloc[0::3].mean()
    jsd_v2 = df_results_wording['JSD_to_Human'].iloc[1::3].mean()
    jsd_v3 = df_results_wording['JSD_to_Human'].iloc[2::3].mean()

    summary_data = {
        "Wording_Variante": ["v1", "v2", "v3"],
        "MAE": [mae_v1, mae_v2, mae_v3],
        "RMSE": [rmse_v1, rmse_v2, rmse_v3],
        "JSD_Mean": [jsd_v1, jsd_v2, jsd_v3]
    }
    summary_df = pd.DataFrame(summary_data)


    return summary_df

    
key_results_wording("bias_results_wording.csv")
    

,Wording_Variante,MAE,RMSE,JSD_Mean
0,v1,0.335722,0.422136,0.221211
1,v2,0.241971,0.295884,0.123067
2,v3,0.264291,0.324506,0.158045


# Evaluation bias calculation

In [6]:
def key_results_bias(dateiname):

    csv_path = os.path.join(repo_root, "results", dateiname)

    df_results_bias = pd.read_csv(csv_path)

    bias = df_results_bias['Bias_Direct']
    mae = bias.abs().mean()
    rmse = np.sqrt((bias ** 2).mean())

    df_results_bias['JSD_to_Human'] = df_results_bias.apply(calculate_jsd_row, axis=1)

    jsd = df_results_bias['JSD_to_Human'].mean()

    summary_data = {
        "MAE": [mae],
        "RMSE": [rmse],
        "JSD_Mean": [jsd],
        "Absolute Bias (mean)": [df_results_bias['Absolute_Bias'].mean()]
    }

    return pd.DataFrame(summary_data)

In [7]:
key_results_bias("bias_results.csv")

,MAE,RMSE,JSD_Mean,Absolute Bias (mean)
0,0.261911,0.323867,0.155243,-1.031786
